In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from gensim.models import Word2Vec, FastText


# 1. Cargar datos
df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")
y = df['t']

columnas_a_evaluar = ['text', 'text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
resultados_w2v = {}

# 2. Función para convertir un informe entero en un solo vector (promediado)
def document_vector(doc_tokens, model):
    palabras_conocidas = [word for word in doc_tokens if word in model.wv.key_to_index]
    if len(palabras_conocidas) == 0:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[palabras_conocidas], axis=0)

# 3. Bucle principal de evaluación
for col in columnas_a_evaluar:
    
    # Convertimos a string por seguridad (evitar errores con nulos) y tokenizamos
    X_texto = df[col].astype(str)
    X_tokens = X_texto.apply(lambda x: x.split())

    # Split (Fuga de datos evitada: separamos antes de entrenar los embeddings)
    X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
        X_tokens, y, test_size=0.20, random_state=42, stratify=y
    )

    # Entrenar Word2Vec solo con el Train de esta columna específica
    w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=100, window=20, min_count=2, workers=6, seed=42)

    # Transformar Train y Test a vectores numéricos
    X_train_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_train_tokens])
    X_test_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_test_tokens])

    # Entrenar y evaluar el clasificador
    modelo_lr = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42)
    modelo_lr.fit(X_train_w2v, y_train)

    score_w2v = f1_score(y_test, modelo_lr.predict(X_test_w2v), average='macro')
    
    # Guardamos el resultado
    resultados_w2v[col] = score_w2v

# 4. Mostrar el ranking final ordenado

for col, score in sorted(resultados_w2v.items(), key=lambda item: item[1], reverse=True):
    print(f" {col.ljust(20)} : {score:.4f} Macro-F1")

 text_stem_nltk       : 0.5121 Macro-F1
 text_lema            : 0.5046 Macro-F1
 text_pos             : 0.4998 Macro-F1
 text_lema_nltk       : 0.4966 Macro-F1
 text_basico          : 0.4953 Macro-F1
 text                 : 0.4677 Macro-F1


In [ ]:
columnas_a_evaluar = ['text', 'text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
resultados_ft = {}

# 1. Función para promediar vectores (La definimos fuera del bucle por eficiencia)
def document_vector_ft(doc_tokens, model):
    palabras_conocidas = [word for word in doc_tokens if word in model.wv]
    if len(palabras_conocidas) == 0:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[palabras_conocidas], axis=0)

# 2. Bucle principal de evaluación
for col in columnas_a_evaluar:
    
    # Aseguramos formato texto y tokenizamos
    X_texto = df[col].astype(str)
    X_tokens = X_texto.apply(lambda x: x.split())

    # Split (Previniendo fuga de datos)
    X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
        X_tokens, y, test_size=0.20, random_state=42, stratify=y
    )

    # Entrenar FastText (Solo con Train)
    ft_model = FastText(sentences=X_train_tokens, vector_size=100, window=20, min_count=2, workers=6, seed=42)

    # Transformar Train y Test a vectores numéricos densos
    X_train_ft = np.array([document_vector_ft(tokens, ft_model) for tokens in X_train_tokens])
    X_test_ft = np.array([document_vector_ft(tokens, ft_model) for tokens in X_test_tokens])

    # Entrenar y evaluar con Regresión Logística
    modelo_lr_ft = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42)
    modelo_lr_ft.fit(X_train_ft, y_train)

    score_ft = f1_score(y_test, modelo_lr_ft.predict(X_test_ft), average='macro')
    
    # Guardar resultado
    resultados_ft[col] = score_ft

# 3. Mostrar el ranking final ordenado

for col, score in sorted(resultados_ft.items(), key=lambda item: item[1], reverse=True):
    print(f" {col.ljust(20)} : {score:.4f} Macro-F1")

 text_lema_nltk       : 0.5139 Macro-F1
 text_stem_nltk       : 0.4954 Macro-F1
 text_basico          : 0.4951 Macro-F1
 text_lema            : 0.4934 Macro-F1
 text                 : 0.4822 Macro-F1
 text_pos             : 0.4763 Macro-F1


In [ ]:
import gensim.downloader as api
columnas_a_evaluar = ['text', 'text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
resultados_glove = {}

glove_model = api.load("glove-wiki-gigaword-100")

# 2. Función para promediar vectores
def document_vector_glove(doc_tokens, model):
    palabras_conocidas = [word for word in doc_tokens if word in model.key_to_index]
    if len(palabras_conocidas) == 0:
        return np.zeros(model.vector_size)
    return np.mean(model[palabras_conocidas], axis=0)

# 3. Bucle principal de evaluación
for col in columnas_a_evaluar:
    
    # Aseguramos formato texto y tokenizamos
    X_texto = df[col].astype(str)
    X_tokens = X_texto.apply(lambda x: x.split())

    # Split (Misma semilla para que sea justo contra los otros modelos)
    X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
        X_tokens, y, test_size=0.20, random_state=42, stratify=y
    )

    # 4. Transformar Train y Test a vectores numéricos (Usando el modelo global)
    X_train_glove = np.array([document_vector_glove(tokens, glove_model) for tokens in X_train_tokens])
    X_test_glove = np.array([document_vector_glove(tokens, glove_model) for tokens in X_test_tokens])

    # 5. Entrenar y evaluar con Regresión Logística (max_iter=3000 para igualar el torneo)
    modelo_lr_glove = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42)
    modelo_lr_glove.fit(X_train_glove, y_train)

    score_glove = f1_score(y_test, modelo_lr_glove.predict(X_test_glove), average='macro')
    
    # Guardar resultado
    resultados_glove[col] = score_glove

# 6. Mostrar el ranking final ordenado


for col, score in sorted(resultados_glove.items(), key=lambda item: item[1], reverse=True):
    print(f" {col.ljust(20)} : {score:.4f} Macro-F1")


 text_lema            : 0.4902 Macro-F1
 text_lema_nltk       : 0.4831 Macro-F1
 text_pos             : 0.4827 Macro-F1
 text_basico          : 0.4763 Macro-F1
 text_stem_nltk       : 0.4714 Macro-F1
 text                 : 0.4268 Macro-F1


In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument


columnas_a_evaluar = ['text', 'text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
resultados_d2v = {}

# Bucle principal de evaluación
for col in columnas_a_evaluar:
    
    # 1. Aseguramos formato texto y tokenizamos
    X_texto = df[col].astype(str)
    X_tokens = X_texto.apply(lambda x: x.split())

    # 2. Split (Previniendo fuga de datos)
    X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
        X_tokens, y, test_size=0.20, random_state=42, stratify=y
    )

    # 3. Formatear datos de entrenamiento a TaggedDocument (Requisito estricto de Doc2Vec)
    # Le ponemos una "etiqueta" numérica a cada historial clínico del train
    train_tagged = [TaggedDocument(words=tokens, tags=[str(i)]) for i, tokens in enumerate(X_train_tokens)]

    # 4. Entrenamos el modelo Doc2Vec
    d2v_model = Doc2Vec(documents=train_tagged, vector_size=100, window=5, min_count=2, workers=6, epochs=20, seed=42)

    # 5. Transformar Train y Test
    X_train_d2v = np.array([d2v_model.infer_vector(tokens) for tokens in X_train_tokens])
    X_test_d2v = np.array([d2v_model.infer_vector(tokens) for tokens in X_test_tokens])

    # 6. Entrenar y evaluar con Regresión Logística
    modelo_lr_d2v = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42)
    modelo_lr_d2v.fit(X_train_d2v, y_train)

    score_d2v = f1_score(y_test, modelo_lr_d2v.predict(X_test_d2v), average='macro')
    
    # Guardar resultado
    resultados_d2v[col] = score_d2v

# 7. Mostrar el ranking final ordenado

for col, score in sorted(resultados_d2v.items(), key=lambda item: item[1], reverse=True):
    print(f" {col.ljust(20)} : {score:.4f} Macro-F1")

 text_lema            : 0.4692 Macro-F1
 text_basico          : 0.4505 Macro-F1
 text_stem_nltk       : 0.4422 Macro-F1
 text_pos             : 0.4310 Macro-F1
 text_lema_nltk       : 0.4276 Macro-F1
 text                 : 0.4138 Macro-F1
